**Environment Setup**

In [1]:
!pip uninstall -yq jax jaxlib jax-cuda12-plugin jax-cuda12-pjrt tensorflow-probability
# heads out since jax might drop support for cuda 12, currently (as of Aug 10, 2026), JAX has issues with CUDA 13
# see this post: https://github.com/jax-ml/jax/issues/37923
!pip install -Uq "jax[cuda12]" tfp-nightly blackjax inference_gym optax

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 148.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 390.9/390.9 kB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 120.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 110.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 184.9/184.9 MB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 MB 29.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires tensorflow-probability>=0.13.0, which is not installed.


Restart runtime and run these checks if anything went wrong:

In [ ]:
# run those checks if package compatibility is in trouble

# import jax
# import tensorflow_probability as tfp
# import jaxlib

# print("jaxlib:", jaxlib.__version__)
# print("TFP:", tfp.__version__)

# !pip show jax
# !pip show jaxlib
# !pip show blackjax

# import jax.numpy as jnp

# import blackjax

# import tensorflow_probability.substrates.jax as tfp
# import inference_gym.using_jax as gym

# print("JAX:", jax.__version__)
# print("BlackJAX:", blackjax.__version__)
# print("TFP:", tfp.__version__)
# print("ArviZ:", avs.__version__)
# print("Inference Gym imported successfully!")

**Package Setup**

In [1]:
# GPU set up to accelerate performance
import os
# in case jax eats up my GPU RAM
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ['XLA_FLAGS'] = (
    '--xla_gpu_triton_gemm_any=True '
    '--xla_gpu_enable_latency_hiding_scheduler=true '
)

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import jax
import jax.numpy as jnp
from jax import random, jit, vmap, lax
import tensorflow_probability.substrates.jax as tfp
tfd = tfp.distributions
import inference_gym.using_jax as gym
import jaxlib
import blackjax

from blackjax.adaptation.base import get_filter_adapt_info_fn
import optax

# import arviz as az
# import arviz_stats as avs

import warnings
warnings.filterwarnings('ignore')

import psutil

import gc

from google.colab import drive
from matplotlib.lines import Line2D

process = psutil.Process(os.getpid())

def mem(msg):
    print(f"{msg}: {process.memory_info().rss / 1024**2:.1f} MB")

# verification to make sure this is on a GPU
print(jax.devices())
print(jax.default_backend())

[CudaDevice(id=0)]
gpu


In [3]:
max_warmup = 1000
warmup_window = 100

window_array = np.append(np.repeat(10, 10),
                      np.repeat(warmup_window, max_warmup // warmup_window - 1))

warmup_length = np.repeat(10, len(window_array))
for i in range(len(warmup_length) - 1):
    warmup_length[i + 1] = warmup_length[i] + window_array[i + 1]

# Transition kernel for short regime
repitition = 10
num_chains_short = 2048
num_super_chains = 16

In [4]:
# quantiles for chi squared with df = 1
chi_up = 3.841459 # 95th quantile for chi squared with df = 1
chi_lo = 0.00393214  # 05th quantile for chi squared with df = 1
tau = 1e-4
M = num_chains_short // num_super_chains
nRhat_lower = np.sqrt(1 + 1 / M + tau)
eps_lower = nRhat_lower - 1
bound = [chi_lo / num_chains_short, chi_up / num_chains_short]
threshold = eps_lower

In [5]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
utility_link = '/content/drive/MyDrive/JHU Stuff/Capstone/Utility_Functions/New_PF_Utility.py'
with open(utility_link) as f: exec(f.read())

**Rosenbrock Example**

In [6]:
target = gym.targets.VectorModel(
    gym.targets.Banana(),
    flatten_sample_transformations=True
)

num_dimensions = target.event_shape[0]

# print("Target:", type(target))
# print("Dimensions:", num_dimensions)
# print("Event shape:", target.event_shape)
# Get some estimates of the mean and variance.
try:
  mean_est = target.sample_transformations['identity'].ground_truth_mean
except:
  print('no ground truth mean')
  mean_est = (result.all_states[num_warmup:, :]).mean(0).mean(0)
try:
  var_est = target.sample_transformations['identity'].ground_truth_standard_deviation**2
except:
  print('no ground truth std dev')
  var_est = ((result.all_states[num_warmup:, :]**2).mean(0).mean(0) -
             mean_est**2)
mean_benchmark = mean_est
var_benchmark = var_est

In [7]:
def target_log_prob_fn(x):
    y = target.default_event_space_bijector(x)
    fldj = target.default_event_space_bijector.forward_log_det_jacobian(x)
    return target.unnormalized_log_prob(y) + fldj
offset = 2.0
init_step_size = 1.
# def initialize(shape, key):
#     return (10 * random.normal(key, shape+(num_dimensions,))+ offset)
# used to be
# initial_state = initialize_fn((num_sub_chains,),key=ranKey)
def initialize(shape, key):
    return (10 * random.normal(key, shape)+ offset)

# make sure to check initialization before running:
# initialize((num_dimensions,), random.PRNGKey(0))
# to make sure it only initialize one position.

In [8]:
#simulation part:
Rosen_MSE_p_list = []
Rosen_RHat_p_list = []
Rosen_state_list_p = []

In [9]:
base_key = random.PRNGKey(0)
keys = random.split(base_key, repitition)
for length in warmup_length:
  mem(f"Simulation Start")
  simulation(length,num_chains_short, num_super_chains,
             initialize, keys, num_dimensions,
             target_log_prob_fn,init_step_size,
             repitition, Rosen_RHat_p_list,Rosen_MSE_p_list,
             mean_benchmark,var_benchmark)

Simulation Start: 1457.5 MB
New Pathfinder Initialization. Warmup Length: 10; mean of MSE is: 0.32784706354141235
Simulation Start: 2198.0 MB
New Pathfinder Initialization. Warmup Length: 20; mean of MSE is: 0.24839036166667938
Simulation Start: 2224.1 MB
New Pathfinder Initialization. Warmup Length: 30; mean of MSE is: 0.18628373742103577
Simulation Start: 2227.5 MB
New Pathfinder Initialization. Warmup Length: 40; mean of MSE is: 0.15294185280799866
Simulation Start: 2228.3 MB
New Pathfinder Initialization. Warmup Length: 50; mean of MSE is: 0.12655586004257202
Simulation Start: 2230.7 MB
New Pathfinder Initialization. Warmup Length: 60; mean of MSE is: 0.10370268672704697
Simulation Start: 2233.2 MB
New Pathfinder Initialization. Warmup Length: 70; mean of MSE is: 0.08695892244577408
Simulation Start: 2241.6 MB
New Pathfinder Initialization. Warmup Length: 80; mean of MSE is: 0.07447005063295364
Simulation Start: 2246.7 MB
New Pathfinder Initialization. Warmup Length: 90; mean of MS

In [10]:
RHat_p_df = pd.DataFrame(Rosen_RHat_p_list)
MSE_p_df = pd.DataFrame(Rosen_MSE_p_list)

In [15]:
MSE_p_df.to_pickle(
    '/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/InitializationExperiment/Rosen_MSE.pkl'
)

RHat_p_df.to_pickle(
    "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/InitializationExperiment/Rosen_R_Hat.pkl"
)

**Bimodal Example**

In [7]:
num_dimensions = 100
offset = 0.0
init_step_size = 1.0

def target_log_prob_fn(x):
    logp1 = (
        jnp.log(0.3)
        - 0.5 * jnp.sum((x + 5.0) ** 2)
        - 0.5 * num_dimensions * jnp.log(2 * jnp.pi)
    )

    logp2 = (
        jnp.log(0.7)
        - 0.5 * jnp.sum((x - 5.0) ** 2)
        - 0.5 * num_dimensions * jnp.log(2 * jnp.pi)
    )

    return jax.scipy.special.logsumexp(
        jnp.array([logp1, logp2])
    )

def initialize (shape, key):
  return 10 * random.normal(key, shape) + offset

# make sure to check initialization before running:
# initialize((num_dimensions,), random.PRNGKey(0))
# to make sure it only initialize one position.

mean_est = jnp.repeat(2, num_dimensions)
var_est = jnp.repeat(22, num_dimensions)
mean_benchmark = mean_est
var_benchmark = var_est

In [8]:
#simulation part:
Bim_MSE_p_list = []
Bim_RHat_p_list = []
Bim_state_list_p = []

In [9]:
base_key = random.PRNGKey(0)
keys = random.split(base_key, repitition)
for length in warmup_length:
  mem(f"Simulation Start")
  simulation(length,num_chains_short, num_super_chains,
             initialize, keys,num_dimensions,
             target_log_prob_fn,init_step_size,
             repitition, Bim_RHat_p_list,Bim_MSE_p_list,
             mean_benchmark,var_benchmark)

Simulation Start: 1454.3 MB
New Pathfinder Initialization. Warmup Length: 10; mean of MSE is: 1.318010926246643
Simulation Start: 2254.8 MB
New Pathfinder Initialization. Warmup Length: 20; mean of MSE is: 1.317840814590454
Simulation Start: 2287.8 MB
New Pathfinder Initialization. Warmup Length: 30; mean of MSE is: 1.3178365230560303
Simulation Start: 2299.2 MB
New Pathfinder Initialization. Warmup Length: 40; mean of MSE is: 1.3179616928100586
Simulation Start: 2310.2 MB
New Pathfinder Initialization. Warmup Length: 50; mean of MSE is: 1.3179104328155518
Simulation Start: 2314.1 MB
New Pathfinder Initialization. Warmup Length: 60; mean of MSE is: 1.3177286386489868
Simulation Start: 2322.4 MB
New Pathfinder Initialization. Warmup Length: 70; mean of MSE is: 1.3180267810821533
Simulation Start: 2326.0 MB
New Pathfinder Initialization. Warmup Length: 80; mean of MSE is: 1.3176789283752441
Simulation Start: 2346.9 MB
New Pathfinder Initialization. Warmup Length: 90; mean of MSE is: 1.31

In [10]:
RHat_p_df = pd.DataFrame(Bim_RHat_p_list)
MSE_p_df = pd.DataFrame(Bim_MSE_p_list)
MSE_p_df.to_pickle(
    "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/InitializationExperiment/Bim_MSE.pkl"
)

RHat_p_df.to_pickle(
    "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/InitializationExperiment/Bim_R_Hat.pkl"
)

**Eight School Example**

In [7]:
# NOTE: inference gym stores the centered parameterization
target_raw = gym.targets.EightSchools()  # store raw to examine doc.
target = gym.targets.VectorModel(target_raw,
                                  flatten_sample_transformations = True)
num_dimensions = target.event_shape[0]
init_step_size = 1.

def initialize (shape, key):
    prior_scale = jnp.append(jnp.array([10., 1.]), jnp.repeat(1., 8))
    prior_offset = jnp.append(jnp.array([0., 5.]), jnp.repeat(0., 8))
    return prior_scale * random.normal(key, shape) + prior_offset

num_schools = 8
y = np.array([28, 8, -3, 7, -1, 1, 18, 12], dtype = np.float32)
sigma = np.array([15, 10, 16, 11, 9, 11, 10, 18], dtype = np.float32)

# NOTE: the reinterpreted batch dimension specifies the dimension of
# each indepdent variable, here the school.
model = tfd.JointDistributionSequential([
    tfd.Normal(loc = 0., scale = 10., name = "mu"),
    tfd.Normal(loc = 5., scale = 1., name = "log_tau"),
    tfd.Independent(tfd.Normal(loc = jnp.zeros(num_schools),
                               scale = jnp.ones(num_schools),
                               name = "eta"),
                    reinterpreted_batch_ndims = 1),
    lambda eta, log_tau, mu: (
        tfd.Independent(tfd.Normal(loc = (mu[..., jnp.newaxis] +
                                        jnp.exp(log_tau[..., jnp.newaxis]) *
                                        eta),
                                   scale = sigma),
                        name = "y",
                        reinterpreted_batch_ndims = 1))
  ])

# minor change from the TFP code
# def target_log_prob_fn(x):
#   mu = x[:, 0]
#   log_tau = x[:, 1]
#   eta = x[:, 2:10]
#   return model.log_prob((mu, log_tau, eta, y))
def target_log_prob_fn(x):
  mu = x[0]
  log_tau = x[1]
  eta = x[2:10]
  return model.log_prob((mu, log_tau, eta, y))

In [8]:
# Use results from running 128 chains with 1000 + 5000 iterations each,
# for non-centered parameterization.
mean_est = np.array([5.8006573 ,  2.4502006 ,  0.6532423 ,  0.09639207,
             -0.23725411,  0.04723661, -0.33556408, -0.19666635,
              0.5390533 ,  0.14633301])

var_est = np.array([29.60382   ,  0.26338503,  0.6383733 ,  0.4928926 ,
              0.65307987,  0.52441144,  0.46658015,  0.5248887 ,
              0.49544162,  0.690975])
mean_benchmark = mean_est
var_benchmark = var_est

In [9]:
#simulation part:
School_MSE_p_list = []
School_RHat_p_list = []
School_state_list_p = []

In [10]:
base_key = random.PRNGKey(0)
keys = random.split(base_key, repitition)
for length in warmup_length:
  mem(f"Simulation Start")
  simulation(length,num_chains_short, num_super_chains,
             initialize, keys,num_dimensions,
             target_log_prob_fn,init_step_size,
             repitition, School_RHat_p_list,School_MSE_p_list,
             mean_benchmark,var_benchmark)

Simulation Start: 1459.8 MB
New Pathfinder Initialization. Warmup Length: 10; mean of MSE is: 1.1758567094802856
Simulation Start: 2201.4 MB
New Pathfinder Initialization. Warmup Length: 20; mean of MSE is: 0.9203706979751587
Simulation Start: 2218.4 MB
New Pathfinder Initialization. Warmup Length: 30; mean of MSE is: 0.6558027863502502
Simulation Start: 2229.2 MB
New Pathfinder Initialization. Warmup Length: 40; mean of MSE is: 0.50709068775177
Simulation Start: 2235.1 MB
New Pathfinder Initialization. Warmup Length: 50; mean of MSE is: 0.3634417653083801
Simulation Start: 2236.3 MB
New Pathfinder Initialization. Warmup Length: 60; mean of MSE is: 0.25456786155700684
Simulation Start: 2241.6 MB
New Pathfinder Initialization. Warmup Length: 70; mean of MSE is: 0.1772049218416214
Simulation Start: 2246.9 MB
New Pathfinder Initialization. Warmup Length: 80; mean of MSE is: 0.12611062824726105
Simulation Start: 2252.0 MB
New Pathfinder Initialization. Warmup Length: 90; mean of MSE is: 0.

In [11]:
RHat_p_df = pd.DataFrame(School_RHat_p_list)
MSE_p_df = pd.DataFrame(School_MSE_p_list)
MSE_p_df.to_pickle(
    "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/InitializationExperiment/School_MSE.pkl"
)

RHat_p_df.to_pickle(
    "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/InitializationExperiment/School_R_Hat.pkl"
)

**Item Response Theory Example**

In [7]:
target = gym.targets.VectorModel(gym.targets.SyntheticItemResponseTheory(),
                                 flatten_sample_transformations=True)
num_dimensions = target.event_shape[0]
init_step_size = 1.

def target_log_prob_fn(x):
  """Unnormalized, unconstrained target density.

  This is a thin wrapper that applies the default bijectors so that we can
  ignore any constraints.
  """
  y = target.default_event_space_bijector(x)
  fldj = target.default_event_space_bijector.forward_log_det_jacobian(x)
  return target.unnormalized_log_prob(y) + fldj

offset = 0
def initialize (shape, key):
  return 10 * random.normal(key, shape) + offset

In [8]:
# Get some estimates of the mean and variance.
try:
  mean_est = target.sample_transformations['identity'].ground_truth_mean
except:
  print('no ground truth mean')
  mean_est = (result.all_states[num_warmup:, :]).mean(0).mean(0)
try:
  var_est = target.sample_transformations['identity'].ground_truth_standard_deviation**2
except:
  print('no ground truth std dev')
  var_est = ((result.all_states[num_warmup:, :]**2).mean(0).mean(0) -
             mean_est**2)

mean_benchmark = mean_est
var_benchmark = var_est

In [9]:
#simulation part:
IRT_MSE_p_list = []
IRT_RHat_p_list = []
IRT_state_list_p = []

In [10]:
base_key = random.PRNGKey(0)
keys = random.split(base_key, repitition)
for length in warmup_length:
  mem(f"Simulation Start")
  simulation(length,num_chains_short, num_super_chains,
             initialize, keys, num_dimensions,
             target_log_prob_fn,init_step_size,
             repitition, IRT_RHat_p_list,IRT_MSE_p_list,
             mean_benchmark,var_benchmark)

Simulation Start: 1458.1 MB
New Pathfinder Initialization. Warmup Length: 10; mean of MSE is: 148908.734375
Simulation Start: 4071.1 MB
New Pathfinder Initialization. Warmup Length: 20; mean of MSE is: 140437.03125
Simulation Start: 4107.0 MB
New Pathfinder Initialization. Warmup Length: 30; mean of MSE is: 138657.296875
Simulation Start: 4160.2 MB
New Pathfinder Initialization. Warmup Length: 40; mean of MSE is: 138463.65625
Simulation Start: 4202.5 MB
New Pathfinder Initialization. Warmup Length: 50; mean of MSE is: 138303.890625
Simulation Start: 4243.9 MB
New Pathfinder Initialization. Warmup Length: 60; mean of MSE is: 138178.53125
Simulation Start: 4287.3 MB
New Pathfinder Initialization. Warmup Length: 70; mean of MSE is: 138097.859375
Simulation Start: 4329.3 MB
New Pathfinder Initialization. Warmup Length: 80; mean of MSE is: 138037.15625
Simulation Start: 4371.3 MB
New Pathfinder Initialization. Warmup Length: 90; mean of MSE is: 137973.65625
Simulation Start: 4412.2 MB
New P

In [11]:
RHat_p_df = pd.DataFrame(IRT_RHat_p_list)
MSE_p_df = pd.DataFrame(IRT_MSE_p_list)
MSE_p_df.to_pickle(
    "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/InitializationExperiment/IRT_MSE.pkl"
)

RHat_p_df.to_pickle(
    "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/InitializationExperiment/IRT_R_Hat.pkl"
)